# ViT-Small + BNNeck для Vehicle Re-ID

Обучение Transformer модели на датасете VeRi-776

**Google Colab + T4 GPU**

## 1. Установка зависимостей

In [1]:
!pip install torch torchvision timm tqdm pyyaml pillow scipy scikit-learn matplotlib kaggle -q

## 2. Скачивание датасета VeRi-776

In [2]:
import kagglehub
import os
import shutil

print('Downloading dataset using kagglehub...')
path_to_dataset_root = kagglehub.dataset_download("abhyudaya12/veri-vehicle-re-identification-dataset")

print(f"Dataset downloaded to: {path_to_dataset_root}")

expected_nested_path = os.path.join(path_to_dataset_root, 'VeRi-776')

if os.path.exists(expected_nested_path):
    veri_776_source_path = expected_nested_path
    print(f"Found nested 'VeRi-776' folder: {veri_776_source_path}")
else:
    veri_776_source_path = path_to_dataset_root
    print(f"'VeRi-776' folder not found directly. Assuming dataset contents are in: {veri_776_source_path}")

veri_776_target_path = '/content/VeRi-776'

if os.path.exists(veri_776_target_path):
    shutil.rmtree(veri_776_target_path)

shutil.copytree(veri_776_source_path, veri_776_target_path)

print('\nДатасет скачан и скопирован в /content/VeRi-776!')
!ls /content/VeRi-776/

100%|██████████| 946M/946M [01:07<00:00, 14.7MB/s] 

Extracting files...


Dataset downloaded to: C:\Users\nik\.cache\kagglehub\datasets\abhyudaya12\veri-vehicle-re-identification-dataset\versions\1
'VeRi-776' folder not found directly. Assuming dataset contents are in: C:\Users\nik\.cache\kagglehub\datasets\abhyudaya12\veri-vehicle-re-identification-dataset\versions\1

Датасет скачан и скопирован в /content/VeRi-776!


'ls' is not recognized as an internal or external command,
operable program or batch file.


## 3. Импорты и проверка GPU

In [3]:
import os
import sys
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from torchvision import transforms
from PIL import Image
import numpy as np
from collections import defaultdict
import random
from tqdm.notebook import tqdm
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
print(f'timm version: {timm.__version__}')

Device: cuda
GPU: NVIDIA GeForce GTX 1660 SUPER
VRAM: 6.0 GB
timm version: 1.0.24


## 4. Конфигурация

In [4]:

DATA_DIR = '/content/VeRi-776/VeRi'

if not os.path.exists(DATA_DIR):
    for folder in os.listdir('/content/'):
        if 'veri' in folder.lower() or 'VeRi' in folder:
            DATA_DIR = f'/content/{folder}'
            break
print(f'DATA_DIR: {DATA_DIR}')

EPOCHS = 50
BATCH_SIZE = 48
P = 12
K = 4
LR = 1e-4
WARMUP_EPOCHS = 10
MARGIN = 0.3
LABEL_SMOOTH = 0.1

VIT_SIZE = 'small'
IMG_SIZE = (224, 224)

SAVE_DIR = '/content/checkpoints_vit'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'Model: ViT-{VIT_SIZE}')
print(f'Batch size: {BATCH_SIZE} (P={P}, K={K})')
print(f'Epochs: {EPOCHS}')

DATA_DIR: /content/VeRi-776/VeRi
Model: ViT-small
Batch size: 48 (P=12, K=4)
Epochs: 50


## 5. Модель ViT + BNNeck

In [5]:
class BNNeck(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.bn = nn.BatchNorm1d(in_features)
        self.bn.bias.requires_grad_(False)
        self.classifier = nn.Linear(in_features, num_classes, bias=False)
        nn.init.normal_(self.bn.weight, 1.0, 0.02)
        nn.init.normal_(self.classifier.weight, std=0.001)

    def forward(self, x):
        bn_x = self.bn(x)
        logits = self.classifier(bn_x)
        return bn_x, logits


class ViTReID(nn.Module):
    def __init__(self, num_classes, model_size='small', pretrained=True):
        super().__init__()
        
        model_names = {
            'tiny': 'vit_tiny_patch16_224',
            'small': 'vit_small_patch16_224',
            'base': 'vit_base_patch16_224'
        }
        model_name = model_names[model_size]
        
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        self.embed_dim = self.backbone.embed_dim
        self.bnneck = BNNeck(self.embed_dim, num_classes)
        
        print(f'ViT: {model_name}, embed_dim={self.embed_dim}')

    def forward(self, x):
        features = self.backbone(x)
        bn_features, logits = self.bnneck(features)
        
        if self.training:
            return features, bn_features, logits
        else:
            return F.normalize(bn_features, p=2, dim=1)

print('Model defined!')

Model defined!


## 6. Функции потерь

In [6]:
class CrossEntropyLabelSmooth(nn.Module):
    def __init__(self, num_classes, epsilon=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, inputs, targets):
        log_probs = self.logsoftmax(inputs)
        targets_one_hot = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
        targets_smooth = (1 - self.epsilon) * targets_one_hot + self.epsilon / self.num_classes
        loss = (-targets_smooth * log_probs).sum(dim=1).mean()
        return loss


class TripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin

    def forward(self, features, labels):
        dist_mat = torch.cdist(features, features, p=2)
        mask_pos = labels.unsqueeze(0) == labels.unsqueeze(1)
        mask_neg = ~mask_pos
        
        dist_pos = dist_mat.clone()
        dist_pos[~mask_pos] = 0
        hard_pos = dist_pos.max(dim=1)[0]
        
        dist_neg = dist_mat.clone()
        dist_neg[~mask_neg] = float('inf')
        hard_neg = dist_neg.min(dim=1)[0]
        
        loss = F.relu(hard_pos - hard_neg + self.margin).mean()
        return loss

print('Losses defined!')

Losses defined!


## 7. Датасет

In [7]:
class VeRiDataset(Dataset):
    def __init__(self, data_dir, split='train', transform=None):
        self.data_dir = data_dir
        self.transform = transform
        
        if split == 'train':
            self.img_dir = os.path.join(data_dir, 'image_train')
            list_file = os.path.join(data_dir, 'name_train.txt')
        elif split == 'query':
            self.img_dir = os.path.join(data_dir, 'image_query')
            list_file = os.path.join(data_dir, 'name_query.txt')
        else:
            self.img_dir = os.path.join(data_dir, 'image_test')
            list_file = os.path.join(data_dir, 'name_test.txt')
        
        self.samples = []
        self.label_to_idx = {}
        self.idx_to_samples = defaultdict(list)
        
        if os.path.exists(list_file):
            with open(list_file, 'r') as f:
                filenames = [line.strip() for line in f if line.strip()]
        else:
            filenames = [f for f in os.listdir(self.img_dir) if f.endswith('.jpg')]
        
        for fname in filenames:
            parts = fname.split('_')
            if len(parts) >= 2:
                vid = int(parts[0])
                cam = int(parts[1][1:]) if parts[1].startswith('c') else 0
                
                if vid not in self.label_to_idx:
                    self.label_to_idx[vid] = len(self.label_to_idx)
                
                label = self.label_to_idx[vid]
                self.samples.append((fname, label, cam))
                self.idx_to_samples[label].append(len(self.samples) - 1)
        
        self.num_classes = len(self.label_to_idx)
        print(f'{split}: {len(self.samples)} images, {self.num_classes} IDs')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fname, label, cam = self.samples[idx]
        img = Image.open(os.path.join(self.img_dir, fname)).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return {'image': img, 'label': label, 'camera_id': cam}


class PKSampler:
    def __init__(self, dataset, p, k):
        self.p, self.k = p, k
        self.idx_to_samples = dataset.idx_to_samples
        self.labels = list(self.idx_to_samples.keys())

    def __iter__(self):
        random.shuffle(self.labels)
        batch = []
        for label in self.labels:
            indices = self.idx_to_samples[label]
            selected = random.sample(indices, self.k) if len(indices) >= self.k else random.choices(indices, k=self.k)
            batch.extend(selected)
            if len(batch) >= self.p * self.k:
                yield batch[:self.p * self.k]
                batch = batch[self.p * self.k:]

    def __len__(self):
        return len(self.labels) // self.p

print('Dataset defined!')

Dataset defined!


## 8. Загрузка данных

In [8]:
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.33)),
])

test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

In [9]:
train_dataset = VeRiDataset(DATA_DIR, split='train', transform=train_transform)
query_dataset = VeRiDataset(DATA_DIR, split='query', transform=test_transform)
gallery_dataset = VeRiDataset(DATA_DIR, split='gallery', transform=test_transform)

NUM_CLASSES = train_dataset.num_classes
print(f'\nTotal classes: {NUM_CLASSES}')

train: 37746 images, 575 IDs
query: 1678 images, 200 IDs
gallery: 11579 images, 200 IDs

Total classes: 575


In [10]:
train_sampler = PKSampler(train_dataset, p=P, k=K)
train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=2, pin_memory=True)
query_loader = DataLoader(query_dataset, batch_size=64, shuffle=False, num_workers=2)
gallery_loader = DataLoader(gallery_dataset, batch_size=64, shuffle=False, num_workers=2)

print(f'Train batches: {len(train_loader)}')

Train batches: 47


## 9. Модель и оптимизатор

In [ ]:
model = ViTReID(num_classes=NUM_CLASSES, model_size=VIT_SIZE, pretrained=True).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f'\nModel: ViT-{VIT_SIZE} + BNNeck')
print(f'Parameters: {num_params:,}')

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

c:\Users\nik\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nik\.cache\huggingface\hub\models--timm--vit_small_patch16_224.augreg_in21k_ft_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


ViT: vit_small_patch16_224, embed_dim=384

Model: ViT-small + BNNeck
Parameters: 21,887,232


In [ ]:
criterion_ce = CrossEntropyLabelSmooth(NUM_CLASSES, epsilon=LABEL_SMOOTH)
criterion_triplet = TripletLoss(margin=MARGIN)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS, eta_min=1e-6)
scaler = GradScaler()

print('Ready to train!')

## 10. Функции оценки

In [ ]:
@torch.no_grad()
def extract_features(model, loader):
    model.eval()
    features, labels, cameras = [], [], []
    for batch in tqdm(loader, desc='Extracting', leave=False):
        imgs = batch['image'].to(device)
        feats = model(imgs)
        features.append(feats.cpu().numpy())
        labels.append(batch['label'].numpy())
        cameras.append(batch['camera_id'].numpy())
    return np.concatenate(features), np.concatenate(labels), np.concatenate(cameras)


def compute_metrics(query_feats, gallery_feats, query_labels, gallery_labels, query_cams, gallery_cams):
    dist_mat = 1 - np.dot(query_feats, gallery_feats.T)
    num_query = len(query_labels)
    all_AP, all_cmc = [], np.zeros(50)
    
    for i in range(num_query):
        q_label, q_cam = query_labels[i], query_cams[i]
        valid_mask = ~((gallery_labels == q_label) & (gallery_cams == q_cam))
        if valid_mask.sum() == 0: continue
        
        distances = dist_mat[i][valid_mask]
        g_labels = gallery_labels[valid_mask]
        indices = np.argsort(distances)
        matches = (g_labels[indices] == q_label).astype(np.int32)
        
        cmc = matches.cumsum()
        cmc[cmc > 1] = 1
        all_cmc[:len(cmc)] += cmc[:50]
        
        num_rel = matches.sum()
        if num_rel > 0:
            precision = matches.cumsum() / (np.arange(len(matches)) + 1)
            all_AP.append((precision * matches).sum() / num_rel)
    
    cmc = all_cmc / num_query * 100
    return {'Rank-1': cmc[0], 'Rank-5': cmc[4], 'Rank-10': cmc[9], 'mAP': np.mean(all_AP) * 100}

## 11. ОБУЧЕНИЕ

In [ ]:
def train_one_epoch(epoch):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    
    for batch_idx, batch in enumerate(pbar):
        if epoch < WARMUP_EPOCHS:
            progress = (epoch * len(train_loader) + batch_idx) / (WARMUP_EPOCHS * len(train_loader))
            for pg in optimizer.param_groups:
                pg['lr'] = LR * progress
        
        imgs = batch['image'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        with autocast():
            features, bn_features, logits = model(imgs)
            loss = criterion_ce(logits, labels) + criterion_triplet(features, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{optimizer.param_groups[0]["lr"]:.6f}'})
    
    if epoch >= WARMUP_EPOCHS:
        scheduler.step()
    return total_loss / len(train_loader)


def evaluate():
    query_feats, query_labels, query_cams = extract_features(model, query_loader)
    gallery_feats, gallery_labels, gallery_cams = extract_features(model, gallery_loader)
    return compute_metrics(query_feats, gallery_feats, query_labels, gallery_labels, query_cams, gallery_cams)

In [ ]:
print('='*60)
print(f'TRAINING ViT-{VIT_SIZE} + BNNeck')
print('='*60)

best_mAP = 0
history = {'loss': [], 'mAP': [], 'rank1': []}

for epoch in range(EPOCHS):
    start_time = time.time()
    avg_loss = train_one_epoch(epoch)
    history['loss'].append(avg_loss)
    
    epoch_time = time.time() - start_time
    print(f'Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f} - Time: {epoch_time:.1f}s')
    
    if (epoch + 1) % 10 == 0 or epoch == EPOCHS - 1:
        print('Evaluating...')
        results = evaluate()
        history['mAP'].append(results['mAP'])
        history['rank1'].append(results['Rank-1'])
        
        print(f"  Rank-1: {results['Rank-1']:.2f}% | Rank-5: {results['Rank-5']:.2f}% | mAP: {results['mAP']:.2f}%")
        
        if results['mAP'] > best_mAP:
            best_mAP = results['mAP']
            torch.save(model.state_dict(), f'{SAVE_DIR}/best_model.pth')
            print(f'  *** New best mAP: {best_mAP:.2f}% ***')

print('\n' + '='*60)
print(f'Training finished! Best mAP: {best_mAP:.2f}%')
print('='*60)

## 12. Финальные результаты

In [ ]:
model.load_state_dict(torch.load(f'{SAVE_DIR}/best_model.pth'))
results = evaluate()

print('\n' + '='*50)
print(f'VIT-{VIT_SIZE.upper()} FINAL RESULTS')
print('='*50)
print(f"Rank-1:  {results['Rank-1']:.2f}%")
print(f"Rank-5:  {results['Rank-5']:.2f}%")
print(f"Rank-10: {results['Rank-10']:.2f}%")
print(f"mAP:     {results['mAP']:.2f}%")
print('='*50)

In [ ]:
import json

final_results = {
    'model': f'ViT-{VIT_SIZE} + BNNeck',
    'dataset': 'VeRi-776',
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'results': {k: float(v) for k, v in results.items()},
    'history': history
}

with open(f'{SAVE_DIR}/results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print(f'Results saved to {SAVE_DIR}/results.json')

## 13. Сохранение на Google Drive

In [ ]:
# Раскомментируй чтобы сохранить
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/checkpoints_vit /content/drive/MyDrive/
# print('Saved to Google Drive!')